# DiffiScape Circuit Solver Benchmark

Head-to-head comparison of three circuit-theory connectivity solvers on random
resistance grids at 100×100, 500×500, and 1000×1000 resolution.

## Solvers

| Solver | Algorithm | Backends | Metric |
|--------|-----------|----------|--------|
| **Torch** | Global Laplacian + uniform absorption, AMG-preconditioned CG (CPU) / Jacobi-preconditioned CG (GPU via CuPy) | CPU, GPU | Current density (single solve) |
| **JAX** | Global Laplacian + boundary absorption, AMJax-preconditioned CG (lineax) | CPU, GPU | Current density (single solve) |
| **Omniscape** | Moving-window per-focal local solves via `scipy.sparse.linalg.spsolve` | CPU only | Per-focal connectivity (many small solves) |

Torch and JAX both solve a single global system of the same size via
AMG-preconditioned CG, but differ in how they handle absorption:

- **Torch**: uniform absorption on all nodes, $(L + \alpha I)v = \mathbf{1}$
- **JAX**: boundary-only absorption, $(L + \alpha I_{\partial})v = \mathbf{1}_{\text{int}}$

This means they produce **correlated but not identical** current density maps.
The speed comparison is still apples-to-apples: same matrix size, same solver
class, same number of unknowns.

Omniscape computes a **different metric** (windowed focal connectivity) via
many small direct solves. It is included as a reference for the computational
cost of the moving-window approach.

## Hardware

Benchmarked via WSL2 on:
- **CPU**: AMD Ryzen / Intel (host CPU)
- **GPU**: NVIDIA RTX 4070 SUPER (12 GB VRAM)

## Reproducibility

All grids use `np.random.RandomState(42)` for deterministic generation.
Timings are the mean of 3 runs after a warmup call.

## Setup

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "True"

import sys
import time
import importlib.util
import numpy as np
import platform

# Add DiffiScape source to path
sys.path.insert(0, os.path.join("..", "inst", "python"))

# --- Torch solver ---
spec = importlib.util.spec_from_file_location(
    "circuit_solver",
    os.path.join("..", "inst", "python", "diff_cs", "03_circuit_solver.py"),
)
cs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cs)

# --- Omniscape solver ---
spec_omni = importlib.util.spec_from_file_location(
    "diff_omniscape",
    os.path.join("..", "inst", "python", "diff_cs", "04_diff_omniscape.py"),
)
omni = importlib.util.module_from_spec(spec_omni)
spec_omni.loader.exec_module(omni)

# --- JAX solver ---
import diffiscape_jax  # patches amjax compat
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from diffiscape_jax.core import prepare_permeability, circuit_solve, circuit_solve_init

print(f"Platform: {platform.platform()}")
print(f"NumPy:    {np.__version__}")
print(f"JAX:      {jax.__version__}")
print(f"Torch GPU (CuPy) available: {cs.gpu_available()}")
try:
    print(f"JAX GPU devices: {jax.devices('gpu')}")
except RuntimeError:
    print("JAX GPU: not available")

## Benchmark Harness

Each solver is timed with 1 warmup call followed by 3 timed runs.
GPU solvers include device synchronization in the timing.

In [ ]:
N_WARMUP = 1
N_RUNS = 3
ABSORPTION = 0.01
GRID_SIZES = [100, 500, 1000]
OMNI_RADIUS = 13
OMNI_BLOCK = 5


def make_grid(size, seed=42):
    """Generate a random resistance grid."""
    return np.random.RandomState(seed).uniform(1.0, 100.0, (size, size)).astype(np.float64)


def time_fn(fn, n_warmup=N_WARMUP, n_runs=N_RUNS):
    """Time a function: warmup, then return mean/std of n_runs in ms."""
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times), np.std(times)

## Part 1: Global Current Density — Torch vs JAX

Both solvers compute current density from a single absorption-grounded
Laplacian solve of the same matrix size ($N \times N$ where $N = \text{rows} \times \text{cols}$).

The key algorithmic difference is where absorption is applied:

- **Torch**: $(L + \alpha I)v = \mathbf{1}$ — uniform absorption everywhere,
  uses `pyamg` (CPU) or Jacobi-preconditioned CG via CuPy (GPU)
- **JAX**: $(L + \alpha I_{\partial})v = \mathbf{1}_{\text{int}}$ — absorption
  only on boundary nodes, source injection only at interior nodes,
  uses AMJax-preconditioned CG via lineax (CPU/GPU)

In [ ]:
results_global = []

for size in GRID_SIZES:
    R = make_grid(size)
    row = {"grid": f"{size}x{size}", "n_cells": size * size}
    print(f"\n--- {size}x{size} ({size*size:,} cells) ---")

    # --- Torch CPU ---
    cs.enable_gpu(False)
    mean_ms, std_ms = time_fn(
        lambda: cs.solve_circuit_absorption(R, absorption=ABSORPTION, output="current")
    )
    row["torch_cpu"] = mean_ms
    row["torch_cpu_std"] = std_ms
    print(f"  Torch CPU:  {mean_ms:>8.1f} ± {std_ms:.1f} ms")

    # --- Torch GPU ---
    try:
        cs.enable_gpu(True)
        mean_ms, std_ms = time_fn(
            lambda: cs.solve_circuit_absorption(R, absorption=ABSORPTION, output="current")
        )
        row["torch_gpu"] = mean_ms
        row["torch_gpu_std"] = std_ms
        print(f"  Torch GPU:  {mean_ms:>8.1f} ± {std_ms:.1f} ms")
        cs.enable_gpu(False)
    except Exception as e:
        row["torch_gpu"] = None
        row["torch_gpu_std"] = None
        print(f"  Torch GPU:  N/A ({e})")

    # --- JAX CPU ---
    cpu_dev = jax.devices("cpu")[0]
    with jax.default_device(cpu_dev):
        perm = jax.device_put(prepare_permeability(jnp.array(R), "resistance"), cpu_dev)
        ss = circuit_solve_init(perm, size, size, absorption=ABSORPTION)
        mean_ms, std_ms = time_fn(
            lambda: circuit_solve(perm, size, size, solver_state=ss)[0].block_until_ready()
        )
    row["jax_cpu"] = mean_ms
    row["jax_cpu_std"] = std_ms
    print(f"  JAX   CPU:  {mean_ms:>8.1f} ± {std_ms:.1f} ms")

    # --- JAX GPU ---
    try:
        gpu_dev = jax.devices("gpu")[0]
        with jax.default_device(gpu_dev):
            perm_g = jax.device_put(prepare_permeability(jnp.array(R), "resistance"), gpu_dev)
            ss_g = circuit_solve_init(perm_g, size, size, absorption=ABSORPTION)
            mean_ms, std_ms = time_fn(
                lambda: circuit_solve(perm_g, size, size, solver_state=ss_g)[0].block_until_ready()
            )
        row["jax_gpu"] = mean_ms
        row["jax_gpu_std"] = std_ms
        print(f"  JAX   GPU:  {mean_ms:>8.1f} ± {std_ms:.1f} ms")
    except Exception as e:
        row["jax_gpu"] = None
        row["jax_gpu_std"] = None
        print(f"  JAX   GPU:  N/A ({e})")

    results_global.append(row)

## Part 2: Omniscape (Moving-Window) — CPU Only

Omniscape uses a fundamentally different approach: for each focal pixel on a
block grid (spacing = 5), it extracts a $(2r+1) \times (2r+1)$ sub-window
(radius = 13 → 27×27 = 729-node sub-grid), grounds the focal pixel, and solves
a local circuit via `scipy.sparse.linalg.spsolve` (direct LU factorization).

This means:
- 100×100 grid → ~400 focal pixels → ~400 small direct solves
- 500×500 grid → ~10,000 focal pixels → ~10,000 solves
- 1000×1000 grid → ~40,000 focal pixels → ~40,000 solves

The Omniscape metric (per-focal connectivity) is different from the global
current density computed by Torch/JAX, so this comparison shows the
**computational cost of the windowed approach**, not a speed comparison
of equivalent algorithms.

In [ ]:
results_omni = []

for size in GRID_SIZES:
    R = make_grid(size)
    n_focal_approx = ((size // OMNI_BLOCK) ** 2)
    print(f"\n--- {size}x{size} ({size*size:,} cells, ~{n_focal_approx} focal solves) ---")

    # Warmup
    if size <= 100:
        omni.solve_diff_omniscape(R, radius=OMNI_RADIUS, block_size=OMNI_BLOCK)

    # Timed run (single run for large grids — Omniscape is slow)
    n_runs = 3 if size <= 100 else 1
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        C_map, _, _ = omni.solve_diff_omniscape(
            R, radius=OMNI_RADIUS, block_size=OMNI_BLOCK
        )
        times.append((time.perf_counter() - t0) * 1000)

    mean_ms = np.mean(times)
    std_ms = np.std(times) if len(times) > 1 else 0.0
    results_omni.append({
        "grid": f"{size}x{size}",
        "n_cells": size * size,
        "n_focal": n_focal_approx,
        "omni_cpu": mean_ms,
        "omni_cpu_std": std_ms,
    })
    print(f"  Omniscape CPU: {mean_ms:>10.1f} ± {std_ms:.1f} ms  ({n_focal_approx} focal solves)")

## Results Summary

In [ ]:
print("=" * 85)
print("GLOBAL CURRENT DENSITY: Torch vs JAX")
print("  Algorithm: (L + α·I)v = b, single solve, absorption = 0.01")
print("=" * 85)
print(f"{'Grid':<12} {'Torch CPU':>12} {'Torch GPU':>12} {'JAX CPU':>12} {'JAX GPU':>12}")
print("-" * 85)
for r in results_global:
    tc = f"{r['torch_cpu']:.1f}ms"
    tg = f"{r['torch_gpu']:.1f}ms" if r.get('torch_gpu') else "N/A"
    jc = f"{r['jax_cpu']:.1f}ms"
    jg = f"{r['jax_gpu']:.1f}ms" if r.get('jax_gpu') else "N/A"
    print(f"{r['grid']:<12} {tc:>12} {tg:>12} {jc:>12} {jg:>12}")

print()
print("=" * 85)
print("OMNISCAPE (MOVING WINDOW): CPU Only")
print(f"  Algorithm: per-focal spsolve, radius={OMNI_RADIUS}, block_size={OMNI_BLOCK}")
print("=" * 85)
print(f"{'Grid':<12} {'Focal solves':>14} {'Omniscape CPU':>15}")
print("-" * 85)
for r in results_omni:
    oc = f"{r['omni_cpu']:.1f}ms"
    if r['omni_cpu'] > 1000:
        oc = f"{r['omni_cpu']/1000:.1f}s"
    print(f"{r['grid']:<12} {r['n_focal']:>14,} {oc:>15}")

print()
print("=" * 85)
print("CROSS-METHOD COMPARISON (best global solve vs Omniscape)")
print("=" * 85)
for rg, ro in zip(results_global, results_omni):
    candidates = [rg["torch_cpu"], rg["jax_cpu"]]
    if rg.get("torch_gpu"): candidates.append(rg["torch_gpu"])
    if rg.get("jax_gpu"): candidates.append(rg["jax_gpu"])
    best_global = min(candidates)
    ratio = ro["omni_cpu"] / best_global
    print(f"  {rg['grid']}: Omniscape is {ratio:.0f}x slower than best global solve")

## Verification: Torch vs JAX with Matched Strategies

When both frameworks use the **same** absorption strategy, their outputs
should be highly correlated. Remaining differences come from edge weighting:
Torch uses harmonic mean of resistance ($w = 2/(R_i+R_j)$), while JAX
uses arithmetic mean of permeability ($w = (p_i+p_j)/2$, where $p = 1/R$).

Reference correlations (100×100):
- **Uniform absorption**: r = 0.714
- **Boundary absorption**: r = 0.896

In [ ]:
R_verify = make_grid(100)

# Torch
cs.enable_gpu(False)
torch_result = cs.solve_circuit_absorption(R_verify, absorption=ABSORPTION, output="current")
torch_cd = torch_result["current_density"]

# JAX
cpu_dev = jax.devices("cpu")[0]
with jax.default_device(cpu_dev):
    perm_v = jax.device_put(prepare_permeability(jnp.array(R_verify), "resistance"), cpu_dev)
    ss_v = circuit_solve_init(perm_v, 100, 100, absorption=ABSORPTION)
    jax_cd, _ = circuit_solve(perm_v, 100, 100, solver_state=ss_v)
    jax_cd = np.array(jax_cd)

corr = np.corrcoef(torch_cd.ravel(), jax_cd.ravel())[0, 1]
print(f"Pearson correlation (Torch vs JAX current density): {corr:.6f}")
print(f"Torch range: [{torch_cd.min():.4f}, {torch_cd.max():.4f}]")
print(f"JAX   range: [{jax_cd.min():.4f}, {jax_cd.max():.4f}]")

## Discussion

### Reference Results (RTX 4070 SUPER, WSL2, 2026-06-27)

#### Global Current Density — Both Absorption Strategies

**Uniform absorption**: $(L + \alpha I)v = \mathbf{1}$ — better conditioned, faster

| Grid | Torch CPU | Torch GPU | JAX CPU | JAX GPU |
|---|---|---|---|---|
| 100×100 | 10.5 ms | **4.2 ms** | 14.5 ms | 58.9 ms |
| 500×500 | 355.2 ms | **19.7 ms** | 319.5 ms | 31.9 ms |
| 1000×1000 | 1540.6 ms | **28.5 ms** | 1573.6 ms | 129.9 ms |

**Boundary absorption**: $(L + \alpha I_{\partial})v = \mathbf{1}_{\text{int}}$ — harder system

| Grid | Torch CPU | Torch GPU | JAX CPU | JAX GPU |
|---|---|---|---|---|
| 100×100 | 23.9 ms | 137.1 ms | **19.9 ms** | 37.9 ms |
| 500×500 | 922.8 ms | 621.2 ms | 547.3 ms | **52.5 ms** |
| 1000×1000 | 3850.6 ms | 1061.4 ms | 2612.3 ms | **222.1 ms** |

#### Omniscape Moving Window (CPU only, radius=13, block=5)

| Grid | Focal solves | Time |
|---|---|---|
| 100×100 | ~400 | 386 ms |
| 500×500 | ~10,000 | 11.5 s |
| 1000×1000 | ~40,000 | 46.7 s |

### Key Findings

**1. Absorption strategy matters more than framework.**
Boundary absorption is 2–2.5× slower than uniform on CPU (both frameworks)
because only boundary nodes get the diagonal conditioning boost.

**2. JAX CPU matches or beats Torch CPU on both strategies.**
Uniform at 500×500: JAX 320 ms vs Torch 355 ms. Boundary at 1000×1000:
JAX 2612 ms vs Torch 3851 ms.

**3. Torch GPU dominates on uniform absorption.**
CuPy's Jacobi-preconditioned CG is 4.6× faster than JAX GPU at 1000×1000
(28.5 ms vs 130 ms) because the preconditioner runs entirely on-device.

**4. Torch GPU struggles on boundary absorption.**
Jacobi preconditioning (diagonal scaling) is a poor fit for the
boundary-only system — at 1000×1000, Torch GPU takes 1061 ms,
which is slower than JAX CPU (2612 ms only 2.5× slower). The CG solver
needs far more iterations without the uniform diagonal boost.

**5. JAX GPU wins decisively on boundary absorption.**
At 1000×1000, JAX GPU (222 ms) is 4.8× faster than Torch GPU (1061 ms)
and 11.8× faster than JAX CPU. AMJax's multigrid preconditioner handles
the harder system much better than Jacobi.

### Why the Preconditioner Matters

| Preconditioner | Uniform α | Boundary α |
|---|---|---|
| **Jacobi** (Torch GPU) | Excellent — α on every diagonal entry makes the system diagonally dominant | Poor — boundary nodes are well-conditioned but interior nodes aren't |
| **AMG** (JAX GPU) | Good — multigrid hierarchy is overkill but still fast | Excellent — algebraic multigrid captures the interior structure |

This explains the reversal: Torch GPU wins 4.6× on uniform absorption but
loses 4.8× on boundary absorption. The preconditioner–problem fit determines
the winner, not the framework.

### When to Use Each

- **Torch GPU + uniform α**: Fastest option when uniform absorption is acceptable
- **JAX GPU + boundary α**: Fastest option when boundary-only grounding is needed
- **JAX CPU**: Competitive with Torch CPU on both strategies; preferred for autodiff pipelines
- **Omniscape**: When per-focal local connectivity is the required metric